# 01 — Exploratory Data Analysis

A thorough look at the Business Entity Resolution data: what each source looks like, how the ground truth is
structured, what kinds of noise separate true matches, what makes non-matches dangerous, how feasible blocking is,
and how the test set (with its unseen **France** slice) differs from train.

The verdict at the end summarises what this means for the pipeline.

**Conventions:** S1 = Source 1 (deduplicated reference), S2/S3 = sources to be matched against it.
Heavy per-row work runs on random samples; everything vectorisable runs on the full data.

## 0. Setup

In [ ]:
# Kaggle bootstrap: fetch the repo code and the packages missing from Kaggle's image (no-op locally)
import os, subprocess
if os.path.exists("/kaggle/input"):
    repo = "/kaggle/working/ZerotoOne_submission"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Swapnil-Pramanik/ZerotoOne_submission.git", repo], check=True)
    subprocess.run(["pip", "install", "-q", "anyascii==0.3.3", "rapidfuzz==3.14.6"], check=True)

In [ ]:
%matplotlib inline
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from anyascii import anyascii
from rapidfuzz import fuzz, process

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.spines.top": False, "axes.spines.right": False})

# repo code lives next to this notebook locally, or in a git clone on Kaggle (see notebooks/kaggle/)
import sys
SRC = next(p for p in (Path.cwd().parent / "code/business_entity_resolution/src",
                       Path("/kaggle/working/ZerotoOne_submission/code/business_entity_resolution/src")) if p.exists())
sys.path.insert(0, str(SRC))
from config import DATA_DIR as DATA, ON_KAGGLE
print(f"data: {DATA}  (on Kaggle: {ON_KAGGLE})")
SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
def read_tsv(path):
    # keep_default_na=False: a business literally named "NA" must stay a string; only empty cells become null
    return pd.read_csv(path, sep="\t", engine="pyarrow", dtype_backend="pyarrow",
                       keep_default_na=False, na_values=[""])


def load_source(split, source):
    df = read_tsv(DATA / split / f"{split}_source{source}.tsv")
    df.columns = ["entity_id", "name", "address", "country"]
    df.insert(0, "split", split)
    df.insert(1, "source", f"S{source}")
    return df


t0 = time.time()
recs = pd.concat([load_source(sp, s) for sp in ("train", "test") for s in (1, 2, 3)], ignore_index=True)
for c in ("split", "source", "country"):
    recs[c] = recs[c].astype("category")
gt = read_tsv(DATA / "train" / "train_ground_truth.tsv")
gt.columns = ["s1", "matched"]
print(f"loaded {len(recs):,} records + {len(gt):,} ground-truth rows in {time.time() - t0:.1f}s, "
      f"{recs.memory_usage(deep=True).sum() / 1e9:.2f} GB")

### Text normalisation used throughout

`normalize`: transliterate non-ASCII text to Latin (`anyascii`, applied only to rows that need it), lowercase,
replace every non-alphanumeric run with a space. This is a deliberately *simple* baseline normaliser — the point
of this notebook is to see how far it gets and where it breaks.

In [ ]:
def normalize(s: pd.Series) -> pd.Series:
    s = s.fillna("")
    nonascii = s.str.contains(r"[^\x00-\x7F]")
    s = s.copy()
    s[nonascii] = [anyascii(x) for x in s[nonascii]]
    return s.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()


t0 = time.time()
recs["name_n"] = normalize(recs["name"])
recs["addr_n"] = normalize(recs["address"])
print(f"normalised in {time.time() - t0:.1f}s")
recs[["source", "country", "name", "name_n", "address", "addr_n"]].sample(6, random_state=SEED)

## 1. Source overview & integrity checks

In [ ]:
g = recs.groupby(["split", "source"], observed=True)
overview = pd.DataFrame({
    "rows": g.size(),
    "unique_ids": g["entity_id"].nunique(),
    "id_prefix_ok": g.apply(lambda d: (d["entity_id"].str.slice(0, 2) == d.name[1]).mean(), include_groups=False),
    "empty_name": g["name"].apply(lambda s: s.isna().mean()),
    "empty_address": g["address"].apply(lambda s: s.isna().mean()),
    "blank_after_norm_name": g["name_n"].apply(lambda s: (s == "").mean()),
})
overview.style.format({"rows": "{:,}", "unique_ids": "{:,}", "id_prefix_ok": "{:.1%}", "empty_name": "{:.2%}",
                       "empty_address": "{:.2%}", "blank_after_norm_name": "{:.3%}"})

In [ ]:
dupe_ids = recs["entity_id"].duplicated().sum()
exact_dupes = recs.duplicated(subset=["split", "source", "name", "address", "country"]).sum()
print(f"entity_ids repeated anywhere across all files: {dupe_ids:,}")
print(f"exact (name, address, country) duplicate rows within a split+source: {exact_dupes:,}")

In [ ]:
recs.groupby(["split", "source"], observed=True).sample(3, random_state=1)[
    ["split", "source", "entity_id", "name", "address", "country"]]

## 2. Country mix

In [ ]:
cmix = pd.crosstab([recs["split"], recs["source"]], recs["country"], normalize="index")
display(cmix.style.format("{:.1%}"))
cmix.plot.barh(stacked=True, figsize=(9, 3.5), title="Country share per split/source").legend(bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# records in S2+S3 per S1 record, by country — a proxy for how many matches/distractors each entity has
counts = recs.groupby(["split", "country", "source"], observed=True).size().unstack("source")
counts["S2+S3 per S1"] = (counts["S2"] + counts["S3"]) / counts["S1"]
counts.style.format({"S1": "{:,}", "S2": "{:,}", "S3": "{:,}", "S2+S3 per S1": "{:.2f}"})

## 3. Missing fields and field shape

In [ ]:
recs["no_addr"] = recs["address"].isna()
recs.pivot_table(index=["split", "source"], columns="country", values="no_addr", aggfunc="mean",
                 observed=True).style.format("{:.2%}")

In [ ]:
recs["name_len"] = recs["name"].str.len()
recs["name_tokens"] = recs["name_n"].str.count(r"\S+")
recs["addr_len"] = recs["address"].str.len()
recs["addr_parts"] = recs["address"].str.count(",") + 1

shape = recs.groupby(["split", "source", "country"], observed=True)[
    ["name_len", "name_tokens", "addr_len", "addr_parts"]].median()
shape

In [ ]:
tr = recs[recs["split"] == "train"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for src in ("S1", "S2", "S3"):
    d = tr[tr["source"] == src]
    axes[0].hist(d["name_len"].clip(upper=80), bins=80, histtype="step", density=True, label=src)
    axes[1].hist(d["name_tokens"].clip(upper=12), bins=np.arange(14) - 0.5, histtype="step", density=True, label=src)
    axes[2].hist(d["addr_len"].dropna().clip(upper=150), bins=75, histtype="step", density=True, label=src)
for ax, t in zip(axes, ["name length (chars, clipped 80)", "name tokens (clipped 12)", "address length (clipped 150)"]):
    ax.set_title(t); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# formatting style: ALL-CAPS and all-lowercase rates
def case_rates(col):
    s = recs[col].fillna("")
    has_alpha = s.str.contains(r"[A-Za-z]")
    return pd.DataFrame({
        f"{col}_upper": (has_alpha & (s == s.str.upper())),
        f"{col}_lower": (has_alpha & (s == s.str.lower())),
    })

cr = pd.concat([recs[["split", "source", "country"]], case_rates("name"), case_rates("address")], axis=1)
case_tbl = cr.groupby(["split", "source", "country"], observed=True).mean()
del cr
case_tbl.style.format("{:.1%}")

## 4. Scripts, transliteration and accents

In [ ]:
SCRIPTS = {
    "Devanagari": r"\p{Devanagari}", "Tamil": r"\p{Tamil}", "Gujarati": r"\p{Gujarati}",
    "Bengali": r"\p{Bengali}", "Telugu": r"\p{Telugu}", "Kannada": r"\p{Kannada}",
    "Malayalam": r"\p{Malayalam}", "Gurmukhi": r"\p{Gurmukhi}", "Oriya": r"\p{Oriya}",
    "Arabic": r"\p{Arabic}", "Latin accents": r"[À-ÖØ-öø-ÿĀ-ž]",
}

rows = []
for col in ("name", "address"):
    s = recs[col].fillna("")
    na = s.str.contains(r"[^\x00-\x7F]")
    sub = recs.loc[na, ["split", "source", "country"]]
    for script, pat in SCRIPTS.items():
        hit = s[na].str.contains(pat)
        r = hit.groupby([sub["split"], sub["source"], sub["country"]], observed=True).sum()
        rows.append(r.rename(f"{col}:{script}"))
    rows.append(na.groupby([recs["split"], recs["source"], recs["country"]], observed=True).sum().rename(f"{col}:any non-ASCII"))

tot = recs.groupby(["split", "source", "country"], observed=True).size()
script_rates = pd.concat(rows, axis=1).fillna(0).div(tot, axis=0)
script_rates = script_rates.loc[:, script_rates.max() > 0.0005]
script_rates.style.format("{:.2%}").background_gradient(axis=None, cmap="Blues")

In [ ]:
# what do non-Latin names/addresses look like, and what does transliteration give us?
india = recs[(recs["country"] == "India") & (recs["split"] == "train")]
ex = india[india["name"].str.contains(r"\p{Devanagari}")].sample(8, random_state=3)
display(ex[["source", "name", "name_n"]])
ex = india[india["address"].fillna("").str.contains(r"\p{Tamil}|\p{Gujarati}|\p{Bengali}|\p{Telugu}|\p{Kannada}")]
ex = ex.sample(min(8, len(ex)), random_state=3)
display(ex[["source", "address", "addr_n"]])

In [ ]:
# where do the local-script fragments sit inside addresses? (mostly the state name?)
loc = india["address"].fillna("")
parts = loc[loc.str.contains(r"[^\x00-\x7F]")].str.split(",").explode().str.strip()
nonlatin_parts = parts[parts.str.contains(r"[^\x00-\x7F]")]
nonlatin_parts.value_counts().head(25).to_frame("count").assign(transliterated=lambda d: [anyascii(x) for x in d.index])

## 5. Ground truth structure

In [ ]:
gt["matched"] = gt["matched"].fillna("")
gt["n_matches"] = np.where(gt["matched"] == "", 0, gt["matched"].str.count(",") + 1)
pairs = (gt.assign(m=gt["matched"].str.split(",")).explode("m")[["s1", "m"]]
           .query("m != ''").reset_index(drop=True))
pairs["m_src"] = pairs["m"].str.slice(0, 2)

train_ids = {s: set(recs.loc[(recs["split"] == "train") & (recs["source"] == s), "entity_id"]) for s in ("S1", "S2", "S3")}
print(f"GT rows: {len(gt):,}   S1 train records: {len(train_ids['S1']):,}")
print(f"every S1 appears in GT exactly once: {set(gt['s1']) == train_ids['S1'] and not gt['s1'].duplicated().any()}")
print(f"positive pairs: {len(pairs):,}  (S2: {(pairs['m_src'] == 'S2').sum():,}, S3: {(pairs['m_src'] == 'S3').sum():,})")
print(f"all matched ids exist in S2/S3: {pairs['m'].isin(train_ids['S2'] | train_ids['S3']).all()}")
print(f"S2/S3 records matched to >1 S1: {pairs['m'].duplicated().sum():,}")
print(f"duplicate ids within a single match list: {pairs.duplicated(['s1', 'm']).sum():,}")

In [ ]:
s1_country = recs.loc[(recs["split"] == "train") & (recs["source"] == "S1")].set_index("entity_id")["country"]
gt["country"] = gt["s1"].map(s1_country)
gt["n_s2"] = gt["matched"].str.count("S2-")
gt["n_s3"] = gt["matched"].str.count("S3-")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
dist = pd.crosstab(gt["n_matches"].clip(upper=10), gt["country"], normalize="columns")
dist.plot.bar(ax=axes[0], title="matches per S1 entity (10 = 10+)")
heat = np.log10(pd.crosstab(gt["n_s2"], gt["n_s3"]) + 1)
im = axes[1].imshow(heat.values, cmap="viridis", origin="lower"); fig.colorbar(im, ax=axes[1])
axes[1].set_title("log10 #S1 entities by (#S2 matches, #S3 matches)")
axes[1].set_xlabel("# S3 matches"); axes[1].set_ylabel("# S2 matches")
plt.tight_layout(); plt.show()

summary = gt.groupby("country", observed=True).agg(
    entities=("s1", "size"), singleton_rate=("n_matches", lambda s: (s == 0).mean()),
    mean_matches=("n_matches", "mean"), median_matches=("n_matches", "median"),
    has_s2=("n_s2", lambda s: (s > 0).mean()), has_s3=("n_s3", lambda s: (s > 0).mean()),
    max_matches=("n_matches", "max"))
summary.style.format({"entities": "{:,}", "singleton_rate": "{:.1%}", "mean_matches": "{:.2f}",
                      "has_s2": "{:.1%}", "has_s3": "{:.1%}"})

In [ ]:
# how many S2/S3 records are distractors (never matched)?
matched_set = set(pairs["m"])
s23 = recs.loc[(recs["split"] == "train") & recs["source"].isin(["S2", "S3"]), ["entity_id", "source", "country", "no_addr", "name_n"]].copy()
s23["is_matched"] = s23["entity_id"].isin(matched_set)
display(s23.pivot_table(index="source", columns="country", values="is_matched", aggfunc="mean", observed=True)
           .style.format("{:.1%}").set_caption("share of S2/S3 records that match some S1"))
print("share of matched vs distractor records with an empty address:")
print(s23.groupby("is_matched")["no_addr"].mean().map("{:.2%}".format).to_string())

In [ ]:
# per-source duplication: how often does a source hold *several* records for the same entity?
multi = pd.DataFrame({
    ">=2 S2 matches": (gt["n_s2"] >= 2).mean(),
    ">=2 S3 matches": (gt["n_s3"] >= 2).mean(),
    "S2 and S3 both": ((gt["n_s2"] > 0) & (gt["n_s3"] > 0)).mean(),
}, index=["share of S1 entities"]).T
multi.style.format("{:.1%}")

## 6. True pairs — how similar are matches really?

In [ ]:
cols = ["entity_id", "name", "address", "country", "name_n", "addr_n"]
train_recs = recs.loc[recs["split"] == "train", cols].set_index("entity_id")
P = (pairs.join(train_recs, on="s1").join(train_recs, on="m", rsuffix="_m"))
print(f"country agreement across ALL {len(P):,} true pairs: {(P['country'].astype(str) == P['country_m'].astype(str)).mean():.4%}")
print(f"identical raw name:            {(P['name'] == P['name_m']).mean():.1%}")
print(f"identical lowercased name:     {(P['name'].str.lower() == P['name_m'].str.lower()).mean():.1%}")
print(f"identical normalised name:     {(P['name_n'] == P['name_n_m']).mean():.1%}")
print(f"identical normalised address:  {(P['addr_n'] == P['addr_n_m']).mean():.1%}")

**Negatives for comparison.** *Random* negatives: a random same-country S2/S3 record per sampled S1 entity.
*Hard* negatives: a random same-country S2/S3 record that shares the **first normalised name token** with the
S1 entity but is not one of its matches — roughly what a naive name block would surface.

In [ ]:
N = 300_000
pos = P.sample(N, random_state=SEED).reset_index(drop=True)
s1_tr = recs.loc[(recs["split"] == "train") & (recs["source"] == "S1"), cols].reset_index(drop=True)
s23_tr = recs.loc[(recs["split"] == "train") & recs["source"].isin(["S2", "S3"]), cols + ["source"]].reset_index(drop=True)
pair_keys = set(zip(pairs["s1"], pairs["m"]))

# random negatives, same country
anchors = s1_tr.sample(N, random_state=SEED + 1).reset_index(drop=True)
neg_parts = []
for ctry, a in anchors.groupby("country", observed=True):
    pool = s23_tr[s23_tr["country"] == ctry]
    pick = pool.iloc[rng.integers(0, len(pool), len(a))].reset_index(drop=True)
    neg_parts.append(pd.concat([a.reset_index(drop=True), pick.add_suffix("_m")], axis=1))
rneg = pd.concat(neg_parts, ignore_index=True)
rneg = rneg[[k not in pair_keys for k in zip(rneg["entity_id"], rneg["entity_id_m"])]]

# hard negatives: share first name token, same country
s23_tr["tok0"] = s23_tr["name_n"].str.extract(r"^(?P<tok>\S*)")["tok"]
anchors["tok0"] = anchors["name_n"].str.extract(r"^(?P<tok>\S*)")["tok"]
pool = s23_tr.sample(frac=1, random_state=SEED).drop_duplicates(["country", "tok0"], keep="first")  # one random per key
pool = pd.concat([pool, s23_tr.sample(frac=1, random_state=SEED + 7).drop_duplicates(["country", "tok0"])])
hneg = anchors.merge(pool.add_suffix("_m"), left_on=["country", "tok0"], right_on=["country_m", "tok0_m"])
hneg = hneg[[k not in pair_keys for k in zip(hneg["entity_id"], hneg["entity_id_m"])]]
hneg = hneg.drop_duplicates("entity_id").sample(min(N, len(hneg)), random_state=SEED)
print(f"positives {len(pos):,}  random negatives {len(rneg):,}  hard negatives {len(hneg):,}")

In [ ]:
def score(df):
    a, b = df["name_n"].tolist(), df["name_n_m"].tolist()
    aa, ba = df["addr_n"].tolist(), df["addr_n_m"].tolist()
    out = pd.DataFrame({
        "name_ratio": process.cpdist(a, b, scorer=fuzz.ratio, workers=-1),
        "name_token_set": process.cpdist(a, b, scorer=fuzz.token_set_ratio, workers=-1),
        "name_token_sort": process.cpdist(a, b, scorer=fuzz.token_sort_ratio, workers=-1),
        "name_partial": process.cpdist(a, b, scorer=fuzz.partial_ratio, workers=-1),
        "addr_token_set": process.cpdist(aa, ba, scorer=fuzz.token_set_ratio, workers=-1),
    }, index=df.index)
    out.loc[(df["addr_n"] == "") | (df["addr_n_m"] == ""), "addr_token_set"] = np.nan
    out["country"] = df["country"].astype(str).values
    return out

S = pd.concat([score(pos).assign(label="positive"), score(rneg).assign(label="random neg"),
               score(hneg).assign(label="hard neg")], ignore_index=True)
feats = ["name_ratio", "name_token_set", "name_token_sort", "name_partial", "addr_token_set"]
S.groupby("label")[feats].quantile([0.05, 0.25, 0.5, 0.75]).unstack().round(0).T.unstack()

In [ ]:
fig, axes = plt.subplots(1, len(feats), figsize=(20, 3.6), sharey=False)
for ax, f in zip(axes, feats):
    for lab in ("positive", "random neg", "hard neg"):
        ax.hist(S.loc[S["label"] == lab, f].dropna(), bins=50, range=(0, 100), histtype="step", density=True, label=lab)
    ax.set_title(f)
axes[0].legend()
plt.tight_layout(); plt.show()

In [ ]:
# per-country view of the two most useful single signals
S.pivot_table(index="country", columns="label", values=["name_token_set", "addr_token_set"], aggfunc="median").round(0)

In [ ]:
# how separable is positive vs HARD negative with the simplest possible rule (both scores above a threshold)?
ph = S[S["label"].isin(["positive", "hard neg"])].copy()
ph["y"] = ph["label"] == "positive"
rows = []
for tn in (60, 70, 80, 90):
    for ta in (50, 60, 70, 80):
        pred = (ph["name_token_set"] >= tn) & (ph["addr_token_set"].fillna(0) >= ta)
        tp = (pred & ph["y"]).sum(); fp = (pred & ~ph["y"]).sum(); fn = (~pred & ph["y"]).sum()
        rows.append({"name>=": tn, "addr>=": ta, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn)})
pd.DataFrame(rows).pivot(index="name>=", columns="addr>=", values=["precision", "recall"]).style.format("{:.1%}") \
  .set_caption("balanced positive vs hard-negative sample — rule-of-thumb only")

## 7. Name noise

In [ ]:
LEGAL = {
    "llc", "inc", "incorporated", "corp", "corporation", "co", "company", "ltd", "limited", "pvt", "private",
    "llp", "plc", "lp", "pllc", "pc", "pa", "public", "sarl", "sas", "sasu", "sa", "sci", "eurl", "snc", "cie", "et",
    "fils", "the", "and", "of", "m", "s", "mr", "mrs", "ms", "sri", "shri", "dr", "services", "group", "enterprises",
}

def relation(r):
    a_raw, b_raw, a, b = r.name, r.name_m, r.name_n, r.name_n_m
    if a_raw == b_raw: return "identical"
    if a_raw.lower() == b_raw.lower(): return "case only"
    if re.search(r"[^\x00-ɏ]", b_raw): return "non-Latin script"
    if a == b: return "punctuation/accents only"
    ta, tb = a.split(), b.split()
    if sorted(ta) == sorted(tb): return "word order only"
    ca, cb = [t for t in ta if t not in LEGAL], [t for t in tb if t not in LEGAL]
    if sorted(ca) == sorted(cb) and ca: return "legal form / filler words"
    if set(ca) <= set(cb) or set(cb) <= set(ca): return "extra/missing core word"
    if fuzz.token_sort_ratio(" ".join(ca), " ".join(cb)) >= 85: return "typo / small edit"
    if re.search(r"\.(com|net|org|in|co|fr|biz)\b", b_raw.lower()): return "domain name"
    return "heavy rewrite"

rel_sample = pos.sample(60_000, random_state=SEED)
rel_sample["relation"] = [relation(r) for r in rel_sample.itertuples()]
rel = pd.crosstab(rel_sample["relation"], rel_sample["country"], normalize="columns")
rel.sort_values(rel.columns[0], ascending=False).style.format("{:.1%}").set_caption("how a true match's name differs from the S1 name")

In [ ]:
for kind in ("heavy rewrite", "typo / small edit", "extra/missing core word", "domain name", "legal form / filler words"):
    print(f"--- {kind}")
    d = rel_sample[rel_sample["relation"] == kind]
    for r in d.sample(min(6, len(d)), random_state=0).itertuples():
        print(f"   {r.name!r:55} -> {r.name_m!r}")

In [ ]:
# junk prefixes / decorations at the start of names
PREFIX_RE = r"""(?i)^\s*(?P<prefix>[-#<>*~_.!@$%^&+=|/:;,'"()\[\]{}?]+|m/s\.?|mr\.?|mrs\.?|dr\.?|sri|shri)(?:\s|$)"""
first = recs["name"].str.extract(PREFIX_RE)["prefix"].str.lower()
junk = (pd.crosstab([recs["split"], recs["source"]], first.fillna("<none>"), normalize="index")
          .drop(columns="<none>"))
junk.loc[:, junk.max().sort_values(ascending=False).index[:15]].style.format("{:.2%}")

In [ ]:
# domain-style names, and legal-form token frequency by source × country (share of names containing each token)
recs["is_domain"] = recs["name"].str.contains(r"(?i)\.(com|net|org|in|co|fr|biz)\b")
display(recs.pivot_table(index=["split", "source"], columns="country", values="is_domain", aggfunc="mean", observed=True)
            .style.format("{:.2%}").set_caption("names that look like domains"))

legal_forms = ["llc", "inc", "incorporated", "corp", "corporation", "co", "company", "ltd", "limited", "pvt",
               "private", "llp", "public", "sarl", "sas", "sasu", "sa", "sci", "eurl", "cie", "fils"]
lf_rows = {}
for (sp, src, c), d in recs.groupby(["split", "source", "country"], observed=True):
    toks = d["name_n"].sample(min(len(d), 200_000), random_state=SEED).str.split(" ").explode()
    vc = toks.value_counts() / min(len(d), 200_000)
    lf_rows[(sp, src, c)] = vc.reindex(legal_forms).fillna(0)
pd.DataFrame(lf_rows).T.style.format("{:.1%}").background_gradient(axis=None, cmap="Blues")

In [ ]:
# most common name tokens per country in S1 — what is "generic" vs "distinctive"?
for c in ("US", "India"):
    d = recs.loc[(recs["split"] == "train") & (recs["source"] == "S1") & (recs["country"] == c), "name_n"]
    print(c, d.str.split(" ").explode().value_counts().head(40).index.tolist())

## 8. Address noise

In [ ]:
PATTERNS = {
    "India PIN (6 digits)": r"\b\d{6}\b",
    "US ZIP at end": r"\b\d{5}(-\d{4})?\s*$",
    "5-digit anywhere": r"\b\d{5}\b",
    "starts with digit": r"^\s*\d",
    "starts with state/region": r"^\s*[A-Za-z]{2}\s*,",
    "landmark (near/opp/behind)": r"(?i)\b(near|nr|opp|opposite|behind|beside)\b",
    "PO box / PMB / suite": r"(?i)\b(p\.?o\.? box|pmb|suite|ste|apt)\b",
    "## / HN / KH prefixes": r"(?i)(##|\bhn\b|\bkh\b|\bh\.?no\b)",
}
addr = recs["address"].fillna("")
has_addr = ~recs["no_addr"]
res = {k: addr.str.contains(p) for k, p in PATTERNS.items()}
res = pd.DataFrame(res)[has_addr]
res.groupby([recs["split"], recs["source"], recs["country"]], observed=True).mean().style.format("{:.1%}") \
   .background_gradient(axis=None, cmap="Blues")

In [ ]:
# street-type spellings per source (US): long vs abbreviated forms
STREET = {"street": ["street", "st"], "road": ["road", "rd"], "avenue": ["avenue", "ave", "av"],
          "drive": ["drive", "dr"], "boulevard": ["boulevard", "blvd"], "lane": ["lane", "ln"], "court": ["court", "ct"]}
rows = {}
for src in ("S1", "S2", "S3"):
    d = recs.loc[(recs["split"] == "train") & (recs["source"] == src) & (recs["country"] == "US"), "addr_n"]
    vc = d.sample(300_000, random_state=SEED).str.split(" ").explode().value_counts()
    rows[src] = {f"{k}: {v}": vc.get(v, 0) for k, vs in STREET.items() for v in vs}
pd.DataFrame(rows).style.format("{:,}")

In [ ]:
# Indian state spelling variants: last comma component of the address
def last_part(s):
    return s.fillna("").str.extract(r"(?P<last>[^,]*)$")["last"].str.strip()

for src in ("S1", "S2", "S3"):
    d = recs.loc[(recs["split"] == "train") & (recs["source"] == src) & (recs["country"] == "India"), "address"]
    print(src, last_part(d.sample(200_000, random_state=SEED)).value_counts().head(18).to_dict())

In [ ]:
# number tokens in addresses are a strong identity signal — how often do true pairs share them?
num_re = re.compile(r"\d+")
def num_stats(df):
    a = [set(num_re.findall(x)) for x in df["addr_n"]]
    b = [set(num_re.findall(x)) for x in df["addr_n_m"]]
    both = np.array([bool(x) and bool(y) for x, y in zip(a, b)])
    share = np.array([bool(x & y) for x, y in zip(a, b)])
    return pd.Series({"both have numbers": both.mean(), "share >=1 number | both have": share[both].mean(),
                      "S1 numbers all present in match | both have": np.mean([x <= y for x, y, z in zip(a, b, both) if z])})

pd.DataFrame({"positive": num_stats(pos), "random neg": num_stats(rneg), "hard neg": num_stats(hneg)}).style.format("{:.1%}")

In [ ]:
# examples of positive pairs with LOW address similarity — what goes wrong?
low = S[(S["label"] == "positive") & (S["addr_token_set"] < 50)].index
d = pos.loc[low[low < len(pos)]].sample(12, random_state=2)
d[["country", "address", "address_m"]]

## 9. Name collisions and look-alikes — the precision risk

F0.5 punishes false merges. The dangerous cases are *distinct* S1 entities that look alike, and distractor S2/S3
records that look like an S1 entity but are not it.

In [ ]:
s1 = recs[(recs["source"] == "S1")].copy()
grp = s1.groupby(["split", "country", "name_n"], observed=True)["entity_id"].transform("size")
s1["name_group"] = grp.values
display(s1.groupby(["split", "country"], observed=True)["name_group"]
          .agg(share_colliding=lambda s: (s > 1).mean(), max_group="max")
          .style.format({"share_colliding": "{:.2%}"}).set_caption("S1 entities whose normalised name is shared with another S1 entity"))
top = s1[(s1["split"] == "train")].groupby(["country", "name_n"], observed=True).size().sort_values(ascending=False).head(15)
top

In [ ]:
# a colliding S1 name group, with the matches each member owns
tr_s1 = s1[(s1["split"] == "train") & (s1["name_group"].between(2, 4))]
for key in tr_s1["name_n"].drop_duplicates().sample(3, random_state=5):
    grp = tr_s1[tr_s1["name_n"] == key]
    print("=" * 110)
    for r in grp.itertuples():
        print(f"{r.entity_id}  {r.name!r}  |  {r.address!r}")
        for m in P.loc[P["s1"] == r.entity_id].itertuples():
            print(f"      -> {m.m}  {m.name_m!r}  |  {m.address_m!r}")

In [ ]:
# distractors: do unmatched S2/S3 records imitate S1 entities? (exact normalised-name hit in same-country S1)
s1_names = s1[s1["split"] == "train"].groupby("country", observed=True)["name_n"].agg(set)
s23["name_in_s1"] = [n in s1_names[c] for n, c in zip(s23["name_n"], s23["country"])]
s23.groupby(["country", "is_matched"], observed=True)["name_in_s1"].mean().unstack().rename(
    columns={False: "distractor", True: "matched"}).style.format("{:.1%}") \
   .set_caption("S2/S3 records whose normalised name exactly equals some same-country S1 name")

In [ ]:
# nearest-name look-alike for a sample of distractors: how close do they get to *some* S1 entity?
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

def nearest_s1_score(country, n=2000):
    s1c = s1[(s1["split"] == "train") & (s1["country"] == country)]
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2, dtype=np.float32)
    X = vec.fit_transform(s1c["name_n"])
    nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(X)
    out = {}
    for lab, flag in (("distractor", False), ("matched", True)):
        q = s23[(s23["country"] == country) & (s23["is_matched"] == flag)].sample(n, random_state=SEED)
        d, _ = nn.kneighbors(vec.transform(q["name_n"]))
        out[lab] = 1 - d[:, 0]
    return out

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, c in zip(axes, ("US", "India")):
    o = nearest_s1_score(c)
    for lab, v in o.items():
        ax.hist(v, bins=40, range=(0, 1), histtype="step", density=True, label=f"{lab} (median {np.median(v):.2f})")
    ax.set_title(f"{c}: cosine sim of name to nearest S1 name (char 3-grams)"); ax.legend()
plt.tight_layout(); plt.show()

## 10. Blocking feasibility

In [ ]:
# document frequency of every name token across train S2+S3, per country
t0 = time.time()
tok = s23_tr[["country", "name_n"]].assign(t=s23_tr["name_n"].str.split(" ")).explode("t")
df_tok = tok.groupby(["country", "t"], observed=True).size()
print(f"token DF table: {len(df_tok):,} (country, token) keys in {time.time() - t0:.1f}s")
df_lookup = {c: df_tok.loc[c].to_dict() for c in df_tok.index.get_level_values(0).unique()}
del tok

In [ ]:
def blocking_signals(df):
    out = []
    for r in df.itertuples():
        a, b = set(r.name_n.split()), set(r.name_n_m.split())
        shared = a & b
        core = shared - LEGAL
        na = set(num_re.findall(r.addr_n)); nb = set(num_re.findall(r.addr_n_m))
        d = df_lookup.get(str(r.country), {})
        out.append({
            "shares any name token": bool(shared),
            "shares core name token": bool(core),
            "shares first token": r.name_n.split(" ")[0] == r.name_n_m.split(" ")[0],
            "shares address number": bool(na & nb),
            "core token OR address number": bool(core) or bool(na & nb),
            "rarest shared core token DF": min((d.get(t, 0) for t in core), default=np.nan),
        })
    return pd.DataFrame(out, index=df.index)

bs_pos = blocking_signals(pos.sample(100_000, random_state=SEED))
bs_neg = blocking_signals(rneg.sample(100_000, random_state=SEED))
flags = [c for c in bs_pos.columns if "DF" not in c]
pd.DataFrame({"recall on true pairs": bs_pos[flags].mean(), "rate on random negatives": bs_neg[flags].mean()}) \
  .style.format("{:.2%}")

In [ ]:
# per-country recall of the best simple key, and the size of the blocks it would imply
bs_pos["country"] = pos.loc[bs_pos.index, "country"].astype(str)
display(bs_pos.groupby("country")[flags].mean().style.format("{:.1%}"))
q = bs_pos["rarest shared core token DF"].dropna()
print("DF of the rarest shared core name token (i.e. block size if you block on that token):")
print(q.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).astype(int).to_string())

In [ ]:
# true pairs that NO simple key catches — what are they?
missed = bs_pos[~bs_pos["core token OR address number"]].index
pos.loc[missed].sample(min(15, len(missed)), random_state=1)[["country", "name", "name_m", "address", "address_m"]]

In [ ]:
# largest blocks: most frequent core tokens (these must be down-weighted / capped)
for c, d in df_lookup.items():
    top = sorted(((v, k) for k, v in d.items() if k not in LEGAL), reverse=True)[:25]
    print(c, [(k, v) for v, k in top])

## 11. Train vs test shift, and the unseen France slice

In [ ]:
recs["devanagari"] = recs["name"].str.contains(r"\p{Devanagari}")
recs["accents"] = recs["name"].str.contains(r"[À-ÖØ-öø-ÿ]")
shift = recs.groupby(["country", "source", "split"], observed=True).agg(
    rows=("entity_id", "size"), no_addr=("no_addr", "mean"), devanagari=("devanagari", "mean"),
    accents=("accents", "mean"), domain=("is_domain", "mean"), name_tokens=("name_tokens", "mean"),
    addr_parts=("addr_parts", "mean"))
shift.style.format({"rows": "{:,}", "no_addr": "{:.2%}", "devanagari": "{:.2%}", "accents": "{:.2%}",
                    "domain": "{:.2%}", "name_tokens": "{:.2f}", "addr_parts": "{:.2f}"})

In [ ]:
fr = recs[recs["country"] == "France"]
fr.groupby("source", observed=True).sample(6, random_state=4)[["source", "entity_id", "name", "address"]]

In [ ]:
for src in ("S1", "S2", "S3"):
    d = fr.loc[fr["source"] == src, "name_n"].str.split(" ").explode().value_counts()
    print(f"{src} name tokens:", d.head(35).index.tolist())
print()
for src in ("S1", "S2", "S3"):
    d = fr.loc[fr["source"] == src, "addr_n"].str.split(" ").explode()
    d = d[~d.str.fullmatch(r"\d+")].value_counts()
    print(f"{src} address tokens:", d.head(35).index.tolist())

In [ ]:
# French address structure: postal codes, street types, region as last component
fa = fr["address"].fillna("")
fp = pd.DataFrame({
    "5-digit postal code": fa.str.contains(r"\b\d{5}\b"),
    "rue/r.": fa.str.contains(r"(?i)\b(rue|r\.)\s"),
    "avenue/av": fa.str.contains(r"(?i)\b(avenue|av\.?)\s"),
    "boulevard/bd": fa.str.contains(r"(?i)\b(boulevard|bd|bld)\b"),
    "bis/ter": fa.str.contains(r"(?i)\b\d+\s*(bis|ter)\b"),
    "starts with region": fa.str.match(r"(?i)^\s*(Île|Ile|Nouvelle|Hauts|Grand|Auvergne|Occitanie|Provence|Bretagne|Normandie|Pays|Centre|Bourgogne|Corse)"),
})
display(fp.groupby(fr["source"], observed=True).mean().style.format("{:.1%}"))
for src in ("S1", "S2", "S3"):
    print(src, last_part(fr.loc[fr["source"] == src, "address"]).value_counts().head(15).to_dict())

In [ ]:
# does the same noise process (junk prefixes, legal-form swaps) seem to apply to France? quick analogue of §7
fr_first = first[fr.index].fillna("<none>")
pd.crosstab(fr["source"], fr_first, normalize="index").drop(columns="<none>", errors="ignore") \
  .pipe(lambda d: d.loc[:, d.max().sort_values(ascending=False).index[:12]]).style.format("{:.2%}")

In [ ]:
# how well does our (transliteration-based) normaliser treat French? accented vs unaccented variants collapse?
ex = fr[fr["accents"]].sample(8, random_state=9)
ex[["source", "name", "name_n", "address", "addr_n"]]

## 12. Leakage sanity checks (IDs and file order)

In [ ]:
# do entity-id numbers or file row positions carry match information? (they should not; we will not use them)
pos_in_file = {}
for src in ("S1", "S2", "S3"):
    ids = recs.loc[(recs["split"] == "train") & (recs["source"] == src), "entity_id"]
    pos_in_file.update(dict(zip(ids, np.arange(len(ids)) / len(ids))))

smp = P.sample(200_000, random_state=SEED)
a_num = smp["s1"].str.slice(3).astype(int); b_num = smp["m"].str.slice(3).astype(int)
a_pos = smp["s1"].map(pos_in_file); b_pos = smp["m"].map(pos_in_file)
print(f"Spearman(id number S1, id number match):   {a_num.corr(b_num, method='spearman'):+.4f}")
print(f"Spearman(file position S1, file position): {a_pos.corr(b_pos, method='spearman'):+.4f}")
gt_order = gt["s1"].map(pos_in_file)
print(f"Spearman(GT row order, S1 file order):     {pd.Series(np.arange(len(gt))).corr(gt_order, method='spearman'):+.4f}")

## 13. Verdict